In [2]:
!pip install nltk pandas openpyxl scikit-learn umap-learn hdbscan

In [3]:
!pip install --upgrade bertopic

In [4]:
!pip install gensim

In [ ]:
# !pip uninstall bertopic
# !pip install bertopic

## **- TOPIC FOR X**

In [5]:
import pandas as pd
from bertopic import BERTopic
import numpy as np
import os
import pickle
import hdbscan
import umap
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import nltk
from nltk.corpus import stopwords
import re
import random
import torch

# SEED
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# PATH
file_tweet = '/content/dataset_lpdp_konten_raw.csv'
output_path = '/content/output_bertopic/'
os.makedirs(output_path, exist_ok=True)

# LOAD DATA
df = pd.read_csv(file_tweet)
df.columns = df.columns.str.strip()

df = df[['doc_id', 'Content']].rename(columns={'Content': 'text_clean'})
df['text_clean'] = df['text_clean'].astype(str)

# CLEANING
def clean_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    return text

df['text_clean'] = df['text_clean'].apply(clean_text)

# CHUNKING + DOC_ID
def chunk_text(text, max_words=80):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

docs = []
doc_ids = []

for _, row in df.iterrows():
    chunks = chunk_text(row['text_clean'])
    for c in chunks:
        docs.append(c)
        doc_ids.append(row['doc_id'])

print("Jumlah chunk:", len(docs))

# STOPWORDS
nltk.download('stopwords', quiet=True)
stopwords_id = stopwords.words('indonesian')

news_stopwords = [
    'menurut','kata','ujar','dalam','akan','bisa','juga',
    'karena','hingga','setelah','seperti','tersebut',
    'bahwa','para','sebuah','lebih','kurang','telah'
]

stopwords_final = stopwords_id + news_stopwords

# EMBEDDING
print("Embedding...")
embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

# COHERENCE PREP
tokenized_docs = [d.split() for d in docs]
dictionary = Dictionary(tokenized_docs)

# VECTORIZER
vectorizer_model = CountVectorizer(
    stop_words=stopwords_final,
    ngram_range=(1,3),
    min_df=3,
    max_df=0.95
)

# UMAP
umap_model = umap.UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# GRID SEARCH
min_cluster_sizes = [50, 100, 150]
min_samples_list = [5, 10]

best_model = None
best_coh = -1
best_params = None
best_topics = None

# TRAINING LOOP
for m in min_cluster_sizes:
    for ms in min_samples_list:

        print(f"\n{'='*50}")
        print(f"min_cluster_size={m}, min_samples={ms}")
        print(f"{'='*50}")

        hdbscan_model = hdbscan.HDBSCAN(
            min_cluster_size=m,
            min_samples=ms,
            metric='euclidean',
            cluster_selection_method='eom'
        )

        model = BERTopic(
            embedding_model=embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            verbose=False
        )

        try:
            topics, _ = model.fit_transform(docs, embeddings)

            # COHERENCE
            topic_words = []
            for tid, tw in model.get_topics().items():
                if tid == -1 or tw is None:
                    continue
                words = [w for w,_ in tw if len(w) > 2]
                if len(words) >= 5:
                    topic_words.append(words)

            if len(topic_words) == 0:
                print("❌ Tidak ada topik valid")
                continue

            cm = CoherenceModel(
                topics=topic_words,
                texts=tokenized_docs,
                dictionary=dictionary,
                coherence='c_v'
            )

            coherence = cm.get_coherence()

            info = model.get_topic_info()
            non_out = info[info['Topic'] != -1]

            if len(non_out) == 0:
                continue

            max_ratio = non_out['Count'].max() / len(docs)

            print(f"Coherence={coherence:.4f} | Topics={len(info)-1} | MaxRatio={max_ratio:.2%}")

            # SAVE BEST
            if coherence > best_coh:
                best_coh = coherence
                best_model = model
                best_params = (m, ms)
                best_topics = topics

        except Exception as e:
            print("❌ Error:", e)
            continue

# FINAL RESULT
print(f"\n🏆 BEST MODEL")
print(f"min_cluster_size={best_params[0]}, min_samples={best_params[1]}")
print(f"Coherence={best_coh:.4f}")

# SAVE MODEL
best_model.get_topic_info().to_excel(
    output_path + 'TOPIC_BEST.xlsx', index=False
)

best_model.save(
    output_path + 'bertopic_best_model',
    serialization="safetensors"
)

# SAVE DATA
with open(output_path + 'data.pkl', 'wb') as f:
    pickle.dump({
        'docs': docs,
        'doc_ids': doc_ids
    }, f)

# SAVE TOPIC PER CHUNK
chunk_df = pd.DataFrame({
    'doc_id': doc_ids,
    'chunk_text': docs,
    'topic': best_topics
})

chunk_df.to_excel(output_path + 'topic_per_chunk.xlsx', index=False)

print("\n✅ Semua output tersimpan di:", output_path)

Jumlah chunk: 6104
Embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/191 [00:00<?, ?it/s]


min_cluster_size=50, min_samples=5
Coherence=0.7888 | Topics=36 | MaxRatio=10.03%

min_cluster_size=50, min_samples=10
Coherence=0.7780 | Topics=37 | MaxRatio=7.22%

min_cluster_size=100, min_samples=5
❌ Error: max_df corresponds to < documents than min_df

min_cluster_size=100, min_samples=10
❌ Error: max_df corresponds to < documents than min_df

min_cluster_size=150, min_samples=5
Coherence=0.8178 | Topics=6 | MaxRatio=40.55%

min_cluster_size=150, min_samples=10
Coherence=0.8162 | Topics=7 | MaxRatio=42.17%

🏆 BEST MODEL
min_cluster_size=150, min_samples=5
Coherence=0.8178

✅ Semua output tersimpan di: /content/output_bertopic/


In [8]:
import pandas as pd

df = pd.read_csv('/content/dataset_lpdp_konten_raw.csv')
df.columns = df.columns.str.strip()

# LOAD HASIL TOPIC PER CHUNK
chunk_df = pd.read_excel('/content/output_bertopic/topic_per_chunk.xlsx')

# MAPPING TOPIK → 4 LABEL
mapping = {
    0: 1,
    2: 2, -1: 2, 5: 2,
    3: 3, 4: 3,
    1: 4
}

label_names = {
    1: "Pendaftaran & Seleksi LPDP",
    2: "Kebijakan & Prioritas Program",
    3: "Kewajiban & Sanksi Penerima",
    4: "Kontroversi Penerima Beasiswa"
}

# apply mapping
chunk_df['label_4'] = chunk_df['topic'].map(mapping)

# buang yang tidak ke-map
chunk_df = chunk_df[chunk_df['label_4'].notna()]

# MAJORITY VOTING PER ARTIKEL
final_label = chunk_df.groupby('doc_id')['label_4'] \
    .agg(lambda x: x.value_counts().index[0]) \
    .reset_index()

# tambah nama label
final_label['label_name'] = final_label['label_4'].map(label_names)

# MERGE
final_df = df.merge(final_label, on='doc_id', how='left')

final_df.to_excel('/content/output_bertopic/hasil_final_4_topik_full.xlsx', index=False)

print("✅ Selesai!")
print("Semua kolom asli tetap ada + label_4 + label_name ditambahkan")

✅ Selesai!
Semua kolom asli tetap ada + label_4 + label_name ditambahkan
